<a href="https://colab.research.google.com/github/jsingh23-umd/nfl-teammate-network-analysis/blob/main/INST414_Module1_NFL_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import networkx as nx

In [3]:
years = range(2015, 2026)

rosters = []

for year in years:
    url = f"https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_{year}.csv"
    year_data = pd.read_csv(url)
    rosters.append(year_data)

nfl_data = pd.concat(rosters, ignore_index=True)

nfl_data.head()

,season,team,position,depth_chart_position,jersey_number,status,full_name,first_name,last_name,birth_date,...,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number
0,2015,SF,K,NaN,9.0,ACT,Phil Dawson,Philip,Dawson,1975-01-23,...,REG,A01,Phil,DAW705989,23860.0,32004441-5770-5989-ac23-bf6cdafcb988,1998.0,1998.0,NaN,NaN
1,2015,IND,QB,NaN,8.0,ACT,Matt Hasselbeck,Matthew,Hasselbeck,1975-09-25,...,REG,I01,Matt,HAS536799,23636.0,32004841-5353-6799-a37b-f4bab15e4312,1998.0,1998.0,GB,187.0
2,2015,DEN,QB,NaN,18.0,ACT,Peyton Manning,Peyton,Manning,1976-03-24,...,SB,A01,Peyton,MAN515097,23446.0,32004d41-4e51-5097-63c8-dfd9cac091f8,1998.0,1998.0,IND,1.0
3,2015,IND,K,NaN,4.0,ACT,Adam Vinatieri,Adam,Vinatieri,1972-12-28,...,REG,A01,Adam,VIN196019,21213.0,32005649-4e19-6019-e626-0b58f9aa81e1,1996.0,1996.0,NaN,NaN
4,2015,OAK,FS,NaN,24.0,ACT,Charles Woodson,Charles,Woodson,1976-10-07,...,REG,A01,Charles,WOO661523,23449.0,3200574f-4f66-1523-6494-7f8d7d48fff1,1998.0,1998.0,OAK,4.0


In [4]:
print(nfl_data.columns.tolist())

['season', 'team', 'position', 'depth_chart_position', 'jersey_number', 'status', 'full_name', 'first_name', 'last_name', 'birth_date', 'height', 'weight', 'college', 'gsis_id', 'espn_id', 'sportradar_id', 'yahoo_id', 'rotowire_id', 'pff_id', 'pfr_id', 'fantasy_data_id', 'sleeper_id', 'years_exp', 'headshot_url', 'ngs_position', 'week', 'game_type', 'status_description_abbr', 'football_name', 'esb_id', 'gsis_it_id', 'smart_id', 'entry_year', 'rookie_year', 'draft_club', 'draft_number']


In [5]:
players = nfl_data[['season', 'team', 'gsis_id', 'full_name', 'position']].copy()

players.head()

,season,team,gsis_id,full_name,position
0,2015,SF,00-0004091,Phil Dawson,K
1,2015,IND,00-0007091,Matt Hasselbeck,QB
2,2015,DEN,00-0010346,Peyton Manning,QB
3,2015,IND,00-0016919,Adam Vinatieri,K
4,2015,OAK,00-0018227,Charles Woodson,FS


In [6]:
# Remove rows missing important information
players = players.dropna(subset=['gsis_id', 'full_name', 'team'])

# Keep each player only once per team and season
players = players.drop_duplicates(
    subset=['season', 'team', 'gsis_id']
)

print("Number of roster records:", len(players))
print("Number of unique players:", players['gsis_id'].nunique())

Number of roster records: 33182
Number of unique players: 9593


In [7]:
g = nx.Graph()

# Add each player as a node
for _, row in players.iterrows():
    g.add_node(
        row['gsis_id'],
        name=row['full_name'],
        position=row['position']
    )

# Group players by team and season
for (season, team), roster in players.groupby(['season', 'team']):
    player_ids = roster['gsis_id'].tolist()

    # Connect every pair of teammates
    for i in range(len(player_ids)):
        for j in range(i + 1, len(player_ids)):
            player1 = player_ids[i]
            player2 = player_ids[j]

            current_weight = g.get_edge_data(
                player1,
                player2,
                default={'weight': 0}
            )['weight']

            g.add_edge(
                player1,
                player2,
                weight=current_weight + 1
            )


print("Number of nodes:", g.number_of_nodes())
print("Number of edges:", g.number_of_edges())

Number of nodes: 9593
Number of edges: 1209393


In [8]:
# Find Peyton Manning's player ID
peyton = players[players['full_name'] == 'Peyton Manning']

peyton[['season', 'team', 'gsis_id', 'full_name']]

,season,team,gsis_id,full_name
2,2015,DEN,00-0010346,Peyton Manning


In [9]:
peyton_id = '00-0010346'

peyton_teammates = list(g.neighbors(peyton_id))

for teammate_id in peyton_teammates[:10]:
    print(
        teammate_id,
        g.nodes[teammate_id]['name'],
        g[peyton_id][teammate_id]['weight']
    )

00-0022793 Antonio Smith 1
00-0023445 DeMarcus Ware 1
00-0023514 Evan Mathis 1
00-0024221 Vernon Davis 1
00-0024313 Owen Daniels 1
00-0025457 Ryan Harris 1
00-0025826 Tyler Polumbus 1
00-0026160 Aqib Talib 1
00-0026237 Andre Caldwell 1
00-0026516 Britton Colquitt 1


In [10]:
# Calculate degree centrality
degree_centrality = nx.degree_centrality(g)

# Get the top 10 players
top_degree = sorted(
    degree_centrality,
    key=degree_centrality.get,
    reverse=True
)[:10]

for player_id in top_degree:
    print(
        g.nodes[player_id]['name'],
        g.nodes[player_id]['position'],
        degree_centrality[player_id],
        "Teammates:", g.degree(player_id)
    )

Josh Johnson QB 0.10341951626355296 Teammates: 992
John Jenkins DL 0.09278565471226022 Teammates: 890
DeAndre Carter WR 0.09049207673060884 Teammates: 868
Nick Vannett TE 0.09049207673060884 Teammates: 868
Ronald Darby DB 0.08726021684737281 Teammates: 837
Raheem Mostert RB 0.08673894912427021 Teammates: 832
Calais Campbell DL 0.08653044203502919 Teammates: 830
Tyrod Taylor QB 0.08653044203502919 Teammates: 830
Eli Apple DB 0.08621768140116763 Teammates: 827
Case Keenum QB 0.0860091743119266 Teammates: 825


In [11]:
josh = players[players['full_name'] == 'Josh Johnson']

josh[['season', 'team', 'position']].sort_values('season')

,season,team,position
339,2015,BUF,QB
340,2015,IND,QB
2484,2016,NYG,QB
3466,2016,JAX,DB
5417,2017,HOU,QB
6118,2017,JAX,DB
8438,2018,WAS,QB
11551,2019,DET,QB
14654,2020,SF,QB
17687,2021,BAL,QB


In [12]:
# Get the player ID for the Josh Johnson who ranked #1
josh_id = top_degree[0]

print("Player ID:", josh_id)
print("Name:", g.nodes[josh_id]['name'])
print("Position:", g.nodes[josh_id]['position'])

# Show only roster records for this exact player
josh_rosters = players[players['gsis_id'] == josh_id]

josh_rosters[['season', 'team', 'position']].sort_values('season')

Player ID: 00-0026300
Name: Josh Johnson
Position: QB


,season,team,position
339,2015,BUF,QB
340,2015,IND,QB
2484,2016,NYG,QB
5417,2017,HOU,QB
8438,2018,WAS,QB
11551,2019,DET,QB
14654,2020,SF,QB
17687,2021,BAL,QB
20640,2022,SF,QB
23764,2023,BAL,QB


In [13]:
# Calculate PageRank
pagerank = nx.pagerank(g)

# Get the top 10 players
top_pagerank = sorted(
    pagerank,
    key=pagerank.get,
    reverse=True
)[:10]

for player_id in top_pagerank:
    print(
        g.nodes[player_id]['name'],
        g.nodes[player_id]['position'],
        pagerank[player_id]
    )

Raheem Mostert RB 0.0003281934540309851
Josh Johnson QB 0.0003270760139485882
Geno Smith QB 0.0003198671833066092
Leonard Williams DL 0.0003145753528571621
Tyler Lockett WR 0.0003144704839560525
Jason Myers K 0.00030998445006038335
Bud Dupree LB 0.00030975819625662893
Denzel Perryman LB 0.00030898915772369285
Lavonte David LB 0.00030876807238826965
Mike Evans WR 0.00030876807238826965


In [14]:
results = []

for player_id in top_degree:
    results.append({
        'Player': g.nodes[player_id]['name'],
        'Position': g.nodes[player_id]['position'],
        'Unique Teammates': g.degree(player_id),
        'Degree Centrality': degree_centrality[player_id]
    })

results_df = pd.DataFrame(results)

results_df

,Player,Position,Unique Teammates,Degree Centrality
0,Josh Johnson,QB,992,0.103420
1,John Jenkins,DL,890,0.092786
2,DeAndre Carter,WR,868,0.090492
3,Nick Vannett,TE,868,0.090492
4,Ronald Darby,DB,837,0.087260
5,Raheem Mostert,RB,832,0.086739
6,Calais Campbell,DL,830,0.086530
7,Tyrod Taylor,QB,830,0.086530
8,Eli Apple,DB,827,0.086218
9,Case Keenum,QB,825,0.086009


In [16]:
final_results = []

for player_id in top_degree:
    final_results.append({
        'Player': g.nodes[player_id]['name'],
        'Position': g.nodes[player_id]['position'],
        'Unique Teammates': g.degree(player_id),
        'Degree Centrality': degree_centrality[player_id],
        'PageRank': pagerank[player_id]
    })

final_results_df = pd.DataFrame(final_results)

final_results_df

,Player,Position,Unique Teammates,Degree Centrality,PageRank
0,Josh Johnson,QB,992,0.103420,0.000327
1,John Jenkins,DL,890,0.092786,0.000304
2,DeAndre Carter,WR,868,0.090492,0.000268
3,Nick Vannett,TE,868,0.090492,0.000282
4,Ronald Darby,DB,837,0.087260,0.000297
5,Raheem Mostert,RB,832,0.086739,0.000328
6,Calais Campbell,DL,830,0.086530,0.000308
7,Tyrod Taylor,QB,830,0.086530,0.000304
8,Eli Apple,DB,827,0.086218,0.000284
9,Case Keenum,QB,825,0.086009,0.000281
